## Dwarf Galaxy SFMS Analysis

**Project Goals:**
Investigatehow dwarf-dwarf interactions and environment influence star star formation. I will look at the galaxies that lie close to the SFMS of (Δlog⁡(sSFR)= −0.25 to 0.25 and those above. 

# Objectives
- Load TNG50 galaxy catalog
- Extract sampole of dwarfs
- Construct SFMS
- Calculate Δlog⁡(sSFR)
- Identify galaxy population above SFMS
- Classify galaxies by environment (central, satellite, backsplash) and record population
- Invesigate whether or not dwarf pairs exihit enhanced star formation


First, I will replicate Figure 1 from: _Bhattacharyya, J., Peter, A. H. G., & Leauthaud, A. (2025, April 23). Dwarf galaxies in the TNG50 field: Connecting their star-formation rates with their environments. arXiv.org. https://arxiv.org/abs/2501.01946_ with my own personal markdowns.

# Import Libraries and Configure Plotting Style

This first cell is responsible for importing python libraries required for the visualization and analysis of the dwarf galaxy sample.

In [ ]:
# Import libraries and simulation utilities

import numpy as np # Main numerical library
import numpy.random as npr 
import matplotlib.pyplot as plt # Plotting library/makes plots
from scipy.spatial import KDTree # KDTree is used for finding nearby objects more efficiently
import matplotlib.colors as mcol
import matplotlib.cm as mcm # Loads Matplotlib colormaps, controls color in plots
import scipy.stats as sts
from matplotlib.lines import Line2D # Loads the Line2D class for creating custom legend elements
from matplotlib.legend import Legend # Loads the Legend class for creating custom legend
import scipy.optimize as sopt
from sim_read import * # Imports all functions from sim_read

plt.rcParams["axes.linewidth"]  = 2
plt.rcParams["xtick.major.size"]  = 8
plt.rcParams["xtick.minor.size"]  = 3
plt.rcParams["ytick.major.size"]  = 8
plt.rcParams["ytick.minor.size"]  = 3
plt.rcParams["xtick.direction"]  = "in"
plt.rcParams["ytick.direction"]  = "in"
plt.rcParams["legend.frameon"] = 'False'
plt.rcParams.update({'font.size': 24})
plt.rcParams.update({'font.family': 'serif'})
plt.rcParams["mathtext.default"] = 'rm'
plt.rcParams["mathtext.fontset"] = 'cm'

# Loads Simulation Data and Computes Derived Mass Properties

This cell is responsible for intializing the TNG50 simulation data by loading host halo and sub halo catalogs, as well as additional group and galaxy properties. It also computes derived mass quantities.

In [ ]:
gls = sim(exgrp_fields=['GroupFirstSub'],exsub_fields=['SubhaloMass','SubhaloStellarPhotometrics','SubhaloMassType'])
sim.mass_add(gls)

# This line creates a selection of host halos that satisfy a minimum mass requirment.

In [ ]:
halo_selc = np.where(gls.hst.group_m200>=10)[0]

## Saves Selected Host Halo Properties to an HDF5 File

This cell is responsible for creating an HDF5 file that contains the properties of host halos that satisfy the minimum halo mass requirment.

In [ ]:
import h5py
hf = h5py.File('tng50_halos_1e10.hdf5', 'w')
 
hf.create_dataset('group_id', data=halo_selc, dtype='int')
hf.create_dataset('group_m200crit', data=gls.hst.group_m200[halo_selc], dtype='float')
hf.create_dataset('group_r200crit', data=gls.hst.Group_R_Crit200[halo_selc]/h0, dtype='float')

print(np.copy(hf.get('group_id')))
print(np.copy(hf.get('group_m200crit')))
print(np.copy(hf.get('group_r200crit')))
print(hf.keys())

hf.close()

## Defines the Galaxy Sample Selection Function

This cell is reposible for defining the function samp_selc(), which constructs the galaxy samples. It also identifies dwarfs and massive galaxies based on stellar mass, incorporates backsplash galaxies from an external catalog, and classifies galaxies according to their enviorment within the simulation.

## Computing the Star-Formation Main Sequence and Select Galaxy Sample

This cell is responsible for computing the SFMS by fitting the relationship between stellar mass and specific star formation rate. The samp_selc() function is called to help formulate the galaxy sample and calculate the enviormntal properties for central dwarfs, backsplash, and satellites. The number of galaxies in each population is printed as a vefication of the sample selection and, a set of marker styles is defined for subsequent figures.

In [ ]:
h0 = 0.677
dconv = 1e-3/h0
mconv = 10 - np.log10(h0)

def samp_selc(floor=7.5,thresh=9.5,NN=5,hfloor=10):
    massive_gal,dwarf_gal = dwf_select(gls.sub.mst,thresh,floor)
    all_gal = np.array([*dwarf_gal,*massive_gal])
    
    rh0 = len(all_gal)/50.0**3
    print(len(all_gal),rh0)

    back_host = np.genfromtxt('backsplash.txt',usecols=0,dtype='int')
    back_dz0 = np.genfromtxt('backsplash.txt',usecols=3,dtype='float')
    back_gal = np.genfromtxt('backsplash.txt',usecols=2,dtype='int')

    dhost = [[],[],[]]
    llg,llt,llh,llr = [[],[],[]],[[],[],[]],[[],[],[]],[[],[],[]]
    x200 = [[],[],[]]

    all_gal_pos =  gls.sub.SubhaloPos[all_gal,:]
    massive_gal_pos = gls.sub.SubhaloPos[massive_gal,:] 

    for j in back_gal:
        lh = gls.sub.SubhaloGrNr[j]
        blh = (np.where(j==back_gal)[0])[0]
        
        if gls.hst.group_m200[lh]>=hfloor:
                #dist, ind = massive_gal_tree.query(gls.sub.SubhaloPos[j,:], k=NN+1)
                rall = wrap_dist_n(massive_gal_pos,gls.sub.SubhaloPos[j,:])
                if gls.sub.mst[j]<9.5: dist,ind = np.sort(rall)[:5],np.argsort(rall)[:5]
                else: dist,ind = np.sort(rall)[1:6],np.argsort(rall)[1:6]
                    
                ind = massive_gal[ind]
                dist *= dconv
                MD_mass = mconv + np.log10(gls.sub.SubhaloMass[ind])
                          
                dhost[2].append(dist[0])
                llt[2].append(max(MD_mass - 3*np.log10(dist)) - 10.96)
                llh[2].append(lh)
                llg[2].append(j)
                x200[2].append(back_dz0[blh]/gls.hst.Group_R_Crit200[back_host[blh]])
                    
                #dist0, ind0 = all_gal_tree.query(gls.sub.SubhaloPos[j,:], k=NN+1)
                rall = wrap_dist_n(all_gal_pos,gls.sub.SubhaloPos[j,:])
                dist0 = np.sort(rall)[:5]
                dist0 *= dconv
                llr[2].append(0.2387324146*(NN+1)/dist0[-1]**3)

        #all_gal = np.delete(all_gal,np.where(all_gal==j)[0])

    all_gal,massive_gal = np.array(list(set(all_gal)-set(back_gal))), np.array(list(set(massive_gal)-set(back_gal)))
    
    all_gal_pos = gls.sub.SubhaloPos[all_gal,:]
    massive_gal_pos = gls.sub.SubhaloPos[massive_gal,:]
    
    all_gal_tree =  KDTree(gls.sub.SubhaloPos[all_gal,:]) 
    massive_gal_tree = KDTree(gls.sub.SubhaloPos[massive_gal,:]) 
    
    
    dwarf_host,host_mem = np.unique(gls.sub.SubhaloGrNr[all_gal],return_inverse=True)
    Ndh = len(dwarf_host)
    for i in range(Ndh):  
        lh = dwarf_host[i]
        #print(gls.hst.group_m200[lh])
        if gls.hst.group_m200[lh]>=hfloor:
            dw_mem = all_gal[np.where(host_mem==i)[0]]

            stellar_sort = np.argsort(gls.sub.mst[dw_mem])
            cen = dw_mem[stellar_sort[-1]]

            #r = wrap_dist_1n(gls.sub.SubhaloPos[cen,:],gls.hst.GroupPos[lh,:])
            if len(dw_mem)>=1:
                rall = wrap_dist_n(massive_gal_pos,gls.sub.SubhaloPos[cen,:])
                if gls.sub.mst[j]<9.5: dist,ind = np.sort(rall)[:5],np.argsort(rall)[:5]
                else: dist,ind = np.sort(rall)[1:6],np.argsort(rall)[1:6]                
                ind = massive_gal[ind]  
                dist *= dconv
                MD_mass = mconv + np.log10(gls.sub.SubhaloMass[ind])

                dhost[0].append(dist[0])
                llt[0].append(max(MD_mass - 3*np.log10(dist)) - 10.96)
                llh[0].append(lh)
                llg[0].append(cen)
                x200[0].append(0)
                
                rall =wrap_dist_n(all_gal_pos,gls.sub.SubhaloPos[cen,:])
                dist0 = np.sort(rall)[:5]
                dist0 *= dconv
                llr[0].append(0.2387324146*(NN+1)/dist0[-1]**3)

                if len(dw_mem)>1:
                    for j in dw_mem[stellar_sort[:-1]]:
                        rall = wrap_dist_n(massive_gal_pos,gls.sub.SubhaloPos[j,:])
                        if gls.sub.mst[j]<9.5: dist,ind = np.sort(rall)[:5],np.argsort(rall)[:5]
                        else: dist,ind = np.sort(rall)[1:6],np.argsort(rall)[1:6] 

                        ind = massive_gal[ind]  
                        dist *= dconv
                        MD_mass = mconv + np.log10(gls.sub.SubhaloMass[ind])
    
                        dhost[1].append(dist[0])
                        llt[1].append(max(MD_mass - 3*np.log10(dist)) - 10.96)
                        llh[1].append(lh)
                        llg[1].append(j)
                        x200[1].append(wrap_dist_1n(gls.sub.SubhaloPos[j,:],gls.hst.GroupPos[lh,:])/gls.hst.Group_R_Crit200[lh])

    
                        rall = wrap_dist_n(all_gal_pos,gls.sub.SubhaloPos[j,:])
                        dist0 = np.sort(rall)[:5]
                        dist0 *= dconv
                        llr[1].append(0.2387324146*(NN+1)/dist0[-1]**3)
                    
        
    return llt,llh,llg,llr,dhost,x200



In [ ]:
sfms_intercept,sfms_slope = dwf_sfms(gls.sub.mst,gls.sub.ssfr)

llt,llh,llg,llr,dhost,x200 = samp_selc(hfloor=9)

N_lll = [len(llh[k]) for k in range(3)]
print(N_lll)
bin_mk = ['o','D','s']

## Combines Galaxy Samples into Unified Arrays

This cell is responsible for merging the central, backsplash, and satellite dwarfs into a unified array, containing their physical and enviormenta properties.

In [ ]:
llg_a = np.array([m for i in range(3) for m in llg[i]])
mst_a = np.array([m for i in range(3) for m in gls.sub.mst[llg[i]]])
llt_a = np.array([t for i in range(3) for t in llt[i]])
x200_a = np.array([t for i in range(3) for t in x200[i]])
llh_a = np.array([h for i in range(3) for h in llh[i]])
dhost_a = np.array([d for i in range(3) for d in dhost[i]])
llr_a = np.log10(np.array([r for i in range(3) for r in llr[i]])) - np.log10(0.142904)
m200_a = gls.hst.group_m200[llh_a]
ssfr_a = np.array([s for i in range(3) for s in gls.sub.ssfr[llg[i]]])
#qnc_a = 1-np.heaviside(ssfr_a-(sfms_intercept -1 + sfms_slope*np.array(mst_a)),1)

x_a = np.array([x for i in range(3) for x in gls.sub.SubhaloPos[llg[i],0]])*dconv - 25
y_a = np.array([y for i in range(3) for y in gls.sub.SubhaloPos[llg[i],1]])*dconv - 25
z_a = np.array([z for i in range(3) for z in gls.sub.SubhaloPos[llg[i],2]])*dconv - 25

## Applying the Final Dwarf Galaxy Sample Selection

This cell is responsible for applying the final selection requirments used to create the sample. It determines the stellar mass of the central galay in each host and then selects only galaxies tht reside in host halos with log(M_200) < 11.5 whose primary galaxy has stellar mass below log(M_* ) < 9.9

In [ ]:
primst_a = np.array([gls.sub.mst[k] for i in range(3) for k in gls.hst.GroupFirstSub[llh[i]]])
dw_samp = np.where((m200_a<11.5)&(primst_a<9.5))[0]
print(len(dw_samp))

## Display of Selected Dwarf Galaxy Sample of Distribution

This cell generates a scatter plot showing the relationship between host halo mass and subhalo stellar mass for the selected sample.

In [ ]:
fig,ax=plt.subplots(figsize=(10,8))

ax.scatter(m200_a,mst_a,c=m200_a,cmap=mcm.Spectral_r,vmin=10,vmax=14,alpha=0.3)
ax.scatter(m200_a[dw_samp],mst_a[dw_samp],c='none',edgecolors='black',marker='o',linewidth=0.5,alpha=0.3)

ax.set_xlim((9,14.5))
ax.set_ylim((7.5,11.5))
ax.axhline(9.5,ls=':',lw=2,color='black')
ax.axvline(11.5,ymax=0.5,ls=':',lw=2,color='black')

ax.text(10.25,10.5,r'$Massive\ Galaxies$',fontsize=30,bbox=dict(facecolor='white',alpha=0.6, boxstyle='round'))
ax.text(11.75,8,r'$Satellite\ Dwarfs$',fontsize=30,bbox=dict(facecolor='white',alpha=0.6, boxstyle='round'))
ax.text(9.55,8,r'$Field\ Dwarfs$',fontsize=30,bbox=dict(facecolor='white',alpha=0.6, boxstyle='round'))

ax.set_xlabel(r'$Host\ log(\mathcal{M}_{200}/M_{\odot})$',fontsize=26)
ax.set_ylabel(r'$Subhalo\ log(\mathcal{M}_{\ast}/M_{\odot})$',fontsize=26)
ax.set_rasterized(True)
#plt.savefig('hostM_subhaloM.pdf',bbox_inches='tight')

# This cell is responsible for computing the difference of the expected sSFR value for each dwarf and the actual (deltalog(sSFR))

In [31]:
sfms_intercept,sfms_slope = dwf_sfms(gls.sub.mst,gls.sub.ssfr)


1494 10829
-8.346120696830761 -0.17679571397785238


In [32]:
sfms_fit = sfms_intercept + sfms_slope * gls.sub.mst # This computes the expected ssfr from the sequence

delta_ssfr = gls.sub.ssfr - sfms_fit # This computes the diffrence from the SFMS
massive_gal,dwarf_gal = dwf_select(gls.sub.mst,thresh = 9.5,floor = 7.5)

1494 10829


/tmp/ipykernel_28063/468612395.py:3: RuntimeWarning: invalid value encountered in subtract
  delta_ssfr = gls.sub.ssfr - sfms_fit # This computes the diffrence from the SFMS


# This cell is responsible for setting a boundary for normal or enhanced ssfr

In [35]:
normal = np.where((delta_ssfr >= -0.25) & (delta_ssfr <= 0.25)) [0]
enhanced = np.where(delta_ssfr > 0.25) [0]

In [45]:
print("Dwarfs:", len(dwarf_gal))
print("Normal:", len(normal))
print("Enhanced:", len(enhanced))

Dwarfs: 10829
Normal: 7078
Enhanced: 2603
